# COMPACT-MI-RFE — Colab Pro
Stage 6 setup. Chạy tuần tự 12 cell. Mặc định chỉ dry run; hai kết quả RF FS16 không chạy lại. Không dùng Test trước khi khóa lựa chọn và refit.


In [1]:
# CELL 1 — Runtime information (read-only; no GPU acceleration assumed)
import os, platform, shutil, sys, importlib.metadata as im
from pathlib import Path
try:
    import psutil
    vm = psutil.virtual_memory()
    ram = {"total": vm.total, "available": vm.available}
    physical = psutil.cpu_count(logical=False)
except ImportError:
    ram, physical = {"total": None, "available": None}, None
disk = shutil.disk_usage("/content")
versions = {}
for name in ("numpy","pandas","scipy","scikit-learn","joblib","psutil"):
    try: versions[name] = im.version(name)
    except im.PackageNotFoundError: versions[name] = None
gpu_tool = shutil.which("nvidia-smi")
print({"os":platform.platform(),"arch":platform.machine(),"python":sys.version,
       "logical_cpus":os.cpu_count(),"physical_cpus":physical,"ram_bytes":ram,
       "disk_free_bytes":disk.free,"gpu_probe_available":bool(gpu_tool),"versions":versions})
print("This scikit-learn pipeline runs on CPU; Colab Pro/High-RAM is not assumed.")

{'os': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'arch': 'x86_64', 'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'logical_cpus': 8, 'physical_cpus': 4, 'ram_bytes': {'total': 54750404608, 'available': 53306941440}, 'disk_free_bytes': 207569371136, 'gpu_probe_available': True, 'versions': {'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'joblib': '1.6.0', 'psutil': '5.9.5'}}
This scikit-learn pipeline runs on CPU; Colab Pro/High-RAM is not assumed.


In [2]:
# CELL 2 — Mount Google Drive (you authorize the Google prompt)
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted:", Path('/content/drive/MyDrive').is_dir())

Mounted at /content/drive
Drive mounted: True


In [6]:

# CELL 3 — Load BOTH datasets from mounted Google Drive
# WINDOWS_CPLUS + CSV_SPLIT_CPLUS
# No download, no copy, no extraction

from pathlib import Path
from google.colab import drive

# 1. Mount Google Drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

# 2. Configure both dataset paths
DRIVE_DATASET_PATH = "/content/drive/MyDrive/WINDOWS_CPLUS"

CSV_SPLIT_DATASET_PATH = "/content/drive/MyDrive/CSV_SPLIT_CPLUS"

# Compatibility with existing Stage 6 notebook
SHARED_FOLDER_ID = ""
DRIVE_ARCHIVE_ID = ""

# 3. Resolve dataset folders
WINDOWS_SOURCE = Path(DRIVE_DATASET_PATH)
CSV_SPLIT_SOURCE = Path(CSV_SPLIT_DATASET_PATH)

# Handle nested directories if present
if (WINDOWS_SOURCE / "WINDOWS_CPLUS").is_dir():
    WINDOWS_SOURCE = WINDOWS_SOURCE / "WINDOWS_CPLUS"

if (CSV_SPLIT_SOURCE / "CSV_SPLIT_CPLUS").is_dir():
    CSV_SPLIT_SOURCE = CSV_SPLIT_SOURCE / "CSV_SPLIT_CPLUS"

# Update paths for downstream cells
DRIVE_DATASET_PATH = str(WINDOWS_SOURCE)
CSV_SPLIT_DATASET_PATH = str(CSV_SPLIT_SOURCE)

# 4. Verify both datasets
print("=== STAGE 6 — DATASET SOURCES ===")

for name, path in [
    ("WINDOWS_CPLUS", WINDOWS_SOURCE),
    ("CSV_SPLIT_CPLUS", CSV_SPLIT_SOURCE),
]:
    print(f"\n{name}")
    print("Path:", path)
    print("Exists:", path.is_dir())

    if not path.is_dir():
        raise FileNotFoundError(
            f"Dataset not found: {path}"
        )

    print("Contents:")
    for item in list(path.iterdir())[:10]:
        print("  ", item.name)

print("\n================================")
print("BOTH DATASETS FOUND: PASS")
print("No dataset copied or downloaded.")
print("================================")


=== STAGE 6 — DATASET SOURCES ===

WINDOWS_CPLUS
Path: /content/drive/MyDrive/WINDOWS_CPLUS
Exists: True
Contents:
   ctu13_sha256_before.json
   input_sha256_before.json
   resume_record.json
   resume_record_attempt_02.json
   reference_stat_samples.csv
   input_sha256_after.json
   ctu13_sha256_after.json
   window_run_summary.json
   progress.json
   FW_512_FS_16

CSV_SPLIT_CPLUS
Path: /content/drive/MyDrive/CSV_SPLIT_CPLUS
Exists: True
Contents:
   Cplus_cutoffs_used.csv
   source_manifest.csv
   split_manifest.csv
   proxy_label_counts_all_roles.csv
   label_mapping.csv
   scenario_counts_all_roles.csv
   validation.json
   test
   val
   train

BOTH DATASETS FOUND: PASS
No dataset copied or downloaded.


In [7]:
# CELL 4 — Clone safely; existing checkout is inspected, never reset/pulled
import subprocess
REPO = Path('/content/RFforNetwork')
URL = 'https://github.com/kasum00/RFforNetwork.git'
if REPO.exists():
    if not (REPO/'.git').is_dir(): raise FileExistsError(f"Not a Git checkout: {REPO}")
    status = subprocess.run(['git','-C',str(REPO),'status','--porcelain'],
                            text=True,capture_output=True,check=True)
    print("Existing checkout; no pull. Local changes:",status.stdout.splitlines())
else:
    subprocess.run(['git','clone','--',URL,str(REPO)],check=True)
required = ['huyen_code/ciciomt24_training.py','huyen_code/ciciomt24_sliding_window.py',
            'huyen_code/ciciomt24_split.py','huyen_code/test_ciciomt24_training.py',
            'huyen_code/test_ciciomt24_split.py','huyen_code/test_ciciomt24_sliding_window.py',
            'colab/experiment_launcher.py','configs/cplus_manifest/split_manifest.csv',
            'CODE_ctu13/[[CTU13]9_ex2_rf.py']
missing = [p for p in required if not (REPO/p).is_file()]
if missing:
    raise RuntimeError(f"GitHub checkout lacks Stage 6/Stage 5 files: {missing}. Publish selected source files manually; do not use an incomplete clone.")
print("Repository ready:",REPO)

Existing checkout; no pull. Local changes: []
Repository ready: /content/RFforNetwork


In [9]:

# CELL 5 — Install dependencies, verify source and run tests
# Compatible with Python 3.12 / 3.13
# No dataset download. No training started.

from pathlib import Path
import importlib.metadata as im
import subprocess
import hashlib
import sys

# ============================================================
# 1. REPOSITORY
# ============================================================

REPO = Path("/content/RFforNetwork").resolve()

if not REPO.is_dir():
    raise FileNotFoundError(REPO)

sys.path.insert(0, str(REPO))

from colab.setup_colab import (
    selected_versions,
    dependency_plan,
    repo_inventory,
)

print("Actual Python:", sys.version)
print("Repository:", REPO)

# ============================================================
# 2. SELECT SCIENTIFIC PROFILE
# ============================================================

expected = selected_versions()
print("\nSelected scientific profile:", expected)

def installed_version(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return None

current = {
    name: installed_version(name)
    for name in expected
}

extras = (
    "psutil",
    "google-api-python-client",
    "google-auth",
)

extras_missing = [
    name for name in extras
    if installed_version(name) is None
]

science_needs = any(
    current[name] != version
    for name, version in expected.items()
)

needs = science_needs or bool(extras_missing)

# ============================================================
# 3. INSTALL ONLY IF NEEDED
# ============================================================

requirements = REPO / "requirements-colab.txt"

if not requirements.is_file():
    raise FileNotFoundError(requirements)

if needs:
    print("\nInstalling missing/mismatched packages...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-r",
            str(requirements),
        ],
        check=True,
    )

    if science_needs:
        raise RuntimeError(
            "Scientific packages installed. "
            "Restart Colab session, then rerun Cells 1-5."
        )

# ============================================================
# 4. VERIFY EXACT SCIENTIFIC VERSIONS
# ============================================================

print("\n=== SCIENTIFIC VERSION CHECK ===")

for name, required in expected.items():
    actual = installed_version(name)

    status = (
        "PASS"
        if actual == required
        else "FAIL"
    )

    print(
        f"{name}: {actual} "
        f"| required={required} | {status}"
    )

    if actual != required:
        raise RuntimeError(
            f"Version mismatch: {name}"
        )

for name in extras:
    if installed_version(name) is None:
        raise RuntimeError(
            f"Missing package: {name}"
        )

# ============================================================
# 5. GLOBAL PIP DEPENDENCY AUDIT
# ============================================================

print("\n=== GLOBAL DEPENDENCY AUDIT ===")

pip_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "check",
    ],
    capture_output=True,
    text=True,
    check=False,
)

GLOBAL_PIP_CHECK_PASSED = (
    pip_result.returncode == 0
)

if GLOBAL_PIP_CHECK_PASSED:
    print("Global pip check: PASS")
else:
    print("WARNING: Global dependency conflicts detected.")
    print(pip_result.stdout)
    print(pip_result.stderr)

    # Known Colab environment conflicts are recorded,
    # not silently treated as a clean environment.
    lines = [
        line.strip()
        for line in pip_result.stdout.splitlines()
        if line.strip()
    ]

    known_prefixes = (
        "ipython ",
        "google-colab ",
    )

    unexpected = [
        line for line in lines
        if not line.startswith(known_prefixes)
    ]

    if unexpected:
        raise RuntimeError(
            "Unexpected dependency conflicts require review:\n"
            + "\n".join(unexpected)
        )

    print(
        "Known Colab conflicts recorded. "
        "Continuing with import and unit tests."
    )

# ============================================================
# 6. FRESH PYTHON IMPORT TEST
# ============================================================

print("\n=== SCIENTIFIC IMPORT TEST ===")

probe = subprocess.run(
    [
        sys.executable,
        "-c",
        """
import numpy
import pandas
import scipy
import sklearn
import joblib
import psutil
import google.auth
import googleapiclient.discovery

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)
print("IMPORT_TEST_PASS")
"""
    ],
    capture_output=True,
    text=True,
)

print(probe.stdout)
print(probe.stderr[-5000:])

if probe.returncode != 0:
    raise RuntimeError(
        "Scientific import test FAILED. "
        "Restart the runtime if packages were recently changed."
    )

print("Scientific imports: PASS")

# ============================================================
# 7. FIX WINDOWS ROOT PATH
# ============================================================

print("\n=== PORTABLE PATH CHECK ===")

training_file = (
    REPO / "huyen_code" / "ciciomt24_training.py"
)

if not training_file.is_file():
    raise FileNotFoundError(training_file)

source = training_file.read_text(encoding="utf-8")

old = 'ROOT = Path(r"D:\\COMPACT-MI-RFE")'
new = 'ROOT = Path(__file__).resolve().parents[1]'

if source.count(old) == 1:
    training_file.write_text(
        source.replace(old, new, 1),
        encoding="utf-8",
    )
    print("Windows ROOT path fixed.")

elif new in source:
    print("ROOT path already fixed.")

else:
    raise RuntimeError(
        "Unexpected ROOT definition. "
        "Inspect ciciomt24_training.py manually."
    )

original_dir = REPO / "CODE_ctu13"

if not original_dir.is_dir():
    raise FileNotFoundError(
        f"Missing CTU-13 scripts: {original_dir}"
    )

print("CTU-13 directory:", original_dir)

# ============================================================
# 8. REPOSITORY INVENTORY
# ============================================================

print("\n=== REPOSITORY INVENTORY ===")

plan = dependency_plan()

if plan["install_needed"]:
    raise RuntimeError(
        f"Scientific versions still differ: {plan}"
    )

inventory = repo_inventory(REPO)

if not inventory["pass"]:
    raise RuntimeError(
        f"Repository inventory failed: {inventory}"
    )

print("Repository inventory: PASS")

# ============================================================
# 9. UNIT TEST HELPER
# ============================================================

def run_tests(label, modules, cwd):
    print(f"\n=== {label} ===")

    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "unittest",
            "-v",
            *modules,
        ],
        cwd=str(cwd),
        capture_output=True,
        text=True,
    )

    print(result.stdout[-4000:])
    print(result.stderr[-8000:])

    if result.returncode != 0:
        raise RuntimeError(
            f"{label} FAILED. Full training blocked."
        )

    print(f"{label}: PASS")

# ============================================================
# 10. SOURCE UNIT TESTS
# ============================================================

run_tests(
    "SOURCE UNIT TESTS",
    [
        "test_ciciomt24_training",
        "test_ciciomt24_split",
        "test_ciciomt24_sliding_window",
    ],
    REPO / "huyen_code",
)

# ============================================================
# 11. STAGE 6 ADAPTER TESTS
# ============================================================

run_tests(
    "STAGE 6 ADAPTER TESTS",
    [
        "colab.test_stage6_colab",
    ],
    REPO,
)

# ============================================================
# 12. SAVE UNIT TEST EVIDENCE
# ============================================================

def file_sha256(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()

UNIT_TEST_EVIDENCE = {
    "passed": True,
    "training_sha256": file_sha256(training_file),
}

print("\n====================================")
print("CELL 5: PASS")
print("SOURCE UNIT TESTS: PASS")
print("STAGE 6 ADAPTER TESTS: PASS")
print("GLOBAL PIP CHECK:", GLOBAL_PIP_CHECK_PASSED)
print(
    "Training SHA256:",
    UNIT_TEST_EVIDENCE["training_sha256"],
)
print("====================================")

print(
    "\nPython 3.13 uses a different NumPy profile "
    "from Stage 5. Bitwise parity is not claimed."
)


Actual Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Repository: /content/RFforNetwork

Selected scientific profile: {'numpy': '2.2.6', 'pandas': '2.3.3', 'scipy': '1.15.3', 'scikit-learn': '1.7.2', 'joblib': '1.5.3'}

=== SCIENTIFIC VERSION CHECK ===
numpy: 2.2.6 | required=2.2.6 | PASS
pandas: 2.3.3 | required=2.3.3 | PASS
scipy: 1.15.3 | required=1.15.3 | PASS
scikit-learn: 1.7.2 | required=1.7.2 | PASS
joblib: 1.5.3 | required=1.5.3 | PASS

=== GLOBAL DEPENDENCY AUDIT ===
ipython 7.34.0 requires jedi, which is not installed.
google-colab 1.0.0 has requirement pandas==2.2.3, but you have pandas 2.3.3.


Known Colab conflicts recorded. Continuing with import and unit tests.

=== SCIENTIFIC IMPORT TEST ===
NumPy: 2.2.6
Pandas: 2.3.3
SciPy: 1.15.3
Scikit-learn: 1.7.2
Joblib: 1.5.3
IMPORT_TEST_PASS


Scientific imports: PASS

=== PORTABLE PATH CHECK ===
Windows ROOT path fixed.
CTU-13 directory: /content/RFforNetwork/CODE_ctu13

=== REPOSITORY INVENTORY ===
Repository inven

In [10]:
# CELL 6 — Prefer the extracted WINDOWS_CPLUS folder; ZIP is optional
import shutil
from colab.drive_manager import discover_shared_dataset, download_archive
from colab.dataset_validator import (WINDOW_ROOT, inspect_archive, prepare_dataset,
    prepare_drive_folder_dataset, prepare_mounted_folder_dataset)
DATASET_READY = False
if WINDOW_ROOT.exists():
    PREPARED = prepare_dataset(REPO, output_root=WINDOW_ROOT)
elif DRIVE_DATASET_PATH:
    source = Path(DRIVE_DATASET_PATH)
    if source.is_dir():
        windows_source = source/'WINDOWS_CPLUS' if (source/'WINDOWS_CPLUS').is_dir() else source
        PREPARED = prepare_mounted_folder_dataset(REPO,windows_source,output_root=WINDOW_ROOT)
    elif source.is_file():
        print('Optional ZIP inventory:',inspect_archive(source))
        PREPARED = prepare_dataset(REPO,archive=source,output_root=WINDOW_ROOT)
    else:
        raise FileNotFoundError(source)
else:
    found = discover_shared_dataset(SHARED_FOLDER_ID)  # user-authorized read-only API
    print('Shared entries:',found['children'])
    print('Audit folder present (not downloaded):',found['audit_folder_present'])
    if found['mode'] == 'folder':
        PREPARED = prepare_drive_folder_dataset(REPO,found['windows_folder']['id'],
                                                found['service'],output_root=WINDOW_ROOT)
    elif found['mode'] == 'zip_available':
        zips = found['zips']
        chosen = next((x for x in zips if x['id']==DRIVE_ARCHIVE_ID),None) if DRIVE_ARCHIVE_ID else (zips[0] if len(zips)==1 else None)
        if chosen is None: raise RuntimeError('Multiple ZIPs: specify DRIVE_ARCHIVE_ID; no folder found.')
        if (Path(chosen['name']).name != chosen['name'] or chr(92) in chosen['name']
                or chosen['name'] in ('.','..')): raise ValueError('Unsafe Drive ZIP filename')
        size = int(chosen.get('size') or 0)
        free = shutil.disk_usage('/content').free
        if size <= 0 or free < 2*size + 8*1024**3:
            raise OSError(f'ZIP disk admission FAIL: need {2*size+8*1024**3:,} bytes; have {free:,}')
        local_zip = Path('/content/datasets/.archive_downloads')/chosen['name']
        if local_zip.exists():
            if local_zip.stat().st_size != size: raise IOError('Existing ZIP differs; inspect manually.')
        else: download_archive(chosen['id'],local_zip)
        PREPARED = prepare_dataset(REPO,archive=local_zip,output_root=WINDOW_ROOT)
    else:
        raise FileNotFoundError('Shared folder has no WINDOWS_CPLUS folder or ZIP')
DATASET_READY = PREPARED['status'] == 'PASS_METADATA_SCHEMA'
print('Dataset path:',WINDOW_ROOT)
print('Configuration folders:',PREPARED['config_directories'],'CSV files:',PREPARED['csv_files'])
print('Manifest counts:',PREPARED['counts'],'Missing files:',PREPARED['missing_files'])
print('Schema validation:',PREPARED['status'],'DATASET_READY=',DATASET_READY)
print('Drive source was not moved or deleted; Test feature rows were not opened.')

[dataset 1/323] transfer FW_256_FS_16/test/Benign/sid001__sid001__Benign_test.pcap_Flow__test__FW256_FS16_windows.csv (2,126,314 bytes)
[dataset 2/323] transfer FW_256_FS_16/test/Benign/sid022__sid022__Benign_train.pcap_Flow__test__FW256_FS16_windows.csv (567,699 bytes)
[dataset 3/323] transfer FW_256_FS_16/test/MQTT-DDoS-Connect_Flood/sid002__sid002__MQTT-DDoS-Connect_Flood_test.pcap_Flow__test__FW256_FS16_windows.csv (238,501,481 bytes)
[dataset 4/323] transfer FW_256_FS_16/test/MQTT-DDoS-Publish_Flood/sid003__sid003__MQTT-DDoS-Publish_Flood_test.pcap_Flow__test__FW256_FS16_windows.csv (804,268 bytes)
[dataset 5/323] transfer FW_256_FS_16/test/MQTT-DDoS-Publish_Flood/sid024__sid024__MQTT-DDoS-Publish_Flood_train.pcap_Flow__test__FW256_FS16_windows.csv (4,648,015 bytes)
[dataset 6/323] transfer FW_256_FS_16/test/MQTT-DoS-Connect_Flood/sid004__sid004__MQTT-DoS-Connect_Flood_test.pcap_Flow__test__FW256_FS16_windows.csv (8,957,118 bytes)
[dataset 7/323] transfer FW_256_FS_16/test/MQTT-Do

AttributeError: 'numpy.ufunc' object has no attribute '__module__' and no __dict__ for setting new attributes

In [13]:

# RECOVER CELL 6 — Reuse all transferred CSVs
# No downloading, no copying, no training

from pathlib import Path
import subprocess
import sys

REPO = Path("/content/RFforNetwork")
FINAL = Path("/content/datasets/CICIoMT24/WINDOWS_CPLUS")
STAGED = Path("/content/datasets/CICIoMT24.incomplete/WINDOWS_CPLUS")

print("=== DATASET RECOVERY ===")
print("Final exists:", FINAL.is_dir())
print("Staged exists:", STAGED.is_dir())

if not FINAL.is_dir() and not STAGED.is_dir():
    raise FileNotFoundError(
        "Neither final nor staged dataset exists. "
        "Do not download again until paths are checked."
    )

# Fix Windows/Linux path interpretation if needed
sw_file = REPO / "huyen_code" / "ciciomt24_sliding_window.py"
source = sw_file.read_text(encoding="utf-8")

old = "Path(segment.output_relative_path).stem"
new = "PureWindowsPath(segment.output_relative_path).stem"

if old in source:
    if source.count(old) != 2:
        raise RuntimeError("Unexpected sliding-window source version")

    import_old = "from pathlib import Path\n"
    if import_old not in source:
        raise RuntimeError("Unexpected pathlib import")

    source = source.replace(
        import_old,
        "from pathlib import Path, PureWindowsPath\n",
        1
    )
    source = source.replace(old, new)
    sw_file.write_text(source, encoding="utf-8")
    print("Portable Windows filename fix: applied")
else:
    print("Portable Windows filename fix: already applied")

# Validate in a fresh Python process
recovery_script = r'''
from pathlib import Path
import os
import sys

REPO = Path("/content/RFforNetwork")
FINAL = Path("/content/datasets/CICIoMT24/WINDOWS_CPLUS")
STAGED = Path("/content/datasets/CICIoMT24.incomplete/WINDOWS_CPLUS")

sys.path.insert(0, str(REPO))

from colab.dataset_validator import validate_dataset

target = FINAL if FINAL.is_dir() else STAGED

csv_files = sum(
    1 for _ in target.rglob("*_windows.csv")
)

print("CSV files found:", csv_files, flush=True)

if csv_files != 322:
    raise RuntimeError(
        f"Expected 322 CSV files, found {csv_files}"
    )

if not (target / "window_run_summary.json").is_file():
    raise FileNotFoundError(
        target / "window_run_summary.json"
    )

print("Validating dataset...", flush=True)

result = validate_dataset(target, REPO)

assert result["status"] == "PASS_METADATA_SCHEMA"
assert result["csv_files"] == 322
assert result["features"] == 380
assert result["config_directories"] == 4

print("Validation: PASS", flush=True)

# Promote only after successful validation
if target == STAGED:
    stage_parent = STAGED.parent
    final_parent = FINAL.parent

    if final_parent.exists():
        raise FileExistsError(
            f"Final destination already exists: {final_parent}"
        )

    os.replace(stage_parent, final_parent)
    print("Staged dataset promoted successfully.")

assert FINAL.is_dir()

print("\n=== FINAL RESULT ===")
print("Status:", result["status"])
print("CSV files:", result["csv_files"])
print("Features:", result["features"])
print("Configurations:", result["config_directories"])
print("Final path:", FINAL)
print("Fingerprint:", result["dataset_fingerprint_sha256"])
print("DATASET_RECOVERY_PASS", flush=True)
'''

process = subprocess.run(
    [sys.executable, "-u", "-c", recovery_script],
    cwd=str(REPO),
    check=False,
)

if process.returncode != 0:
    raise RuntimeError(
        "Recovery validation failed. "
        "Transferred files were not deleted. "
        "Inspect the traceback before retrying."
    )

WINDOW_ROOT = FINAL
DATASET_READY = True

print("\nDATASET_READY =", DATASET_READY)
print("No dataset files downloaded or copied.")


=== DATASET RECOVERY ===
Final exists: False
Staged exists: True
Portable Windows filename fix: applied

DATASET_READY = True
No dataset files downloaded or copied.


In [15]:

# CELL 7 — Validate C+ dataset in a clean Python process
# Avoid broken NumPy state in the current Colab kernel
# No download, no copy, no training

from pathlib import Path
import subprocess
import sys
import json

REPO = Path("/content/RFforNetwork")
WINDOW_ROOT = Path(
    "/content/datasets/CICIoMT24/WINDOWS_CPLUS"
)

if not WINDOW_ROOT.is_dir():
    raise FileNotFoundError(WINDOW_ROOT)

print("=== STAGE 6 DATASET AUDIT ===")
print("Dataset:", WINDOW_ROOT)
print("Running validation in isolated Python process...")

worker = r"""
import sys
import json
from pathlib import Path

repo = Path(sys.argv[1])
window_root = Path(sys.argv[2])

sys.path.insert(0, str(repo))

from colab.dataset_validator import validate_dataset

result = validate_dataset(window_root, repo)

print(
    "__AUDIT_JSON__" +
    json.dumps(result, default=str, ensure_ascii=False)
)
"""

process = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        worker,
        str(REPO),
        str(WINDOW_ROOT),
    ],
    cwd=str(REPO),
    capture_output=True,
    text=True,
)

if process.returncode != 0:
    print(process.stdout[-4000:])
    print(process.stderr[-8000:])
    raise RuntimeError("Dataset audit failed")

marker = "__AUDIT_JSON__"

payload = next(
    (
        line[len(marker):]
        for line in process.stdout.splitlines()
        if line.startswith(marker)
    ),
    None,
)

if payload is None:
    raise RuntimeError("Audit JSON not returned")

DATA_AUDIT = json.loads(payload)

DATASET_READY = (
    DATA_AUDIT.get("status") == "PASS_METADATA_SCHEMA"
    and DATA_AUDIT.get("csv_files") == 322
    and DATA_AUDIT.get("features") == 380
    and DATA_AUDIT.get("config_directories") == 4
)

print("\n=== VALIDATION RESULTS ===")
print("Status:", DATA_AUDIT.get("status"))
print("CSV files:", DATA_AUDIT.get("csv_files"))
print("Features:", DATA_AUDIT.get("features"))
print("Configurations:", DATA_AUDIT.get("config_directories"))
print("Counts:", DATA_AUDIT.get("counts"))
print("Label origin:", DATA_AUDIT.get("label_origin"))
print(
    "Test feature rows read:",
    DATA_AUDIT.get("test_feature_rows_read")
)
print("DATASET_READY:", DATASET_READY)

if not DATASET_READY:
    raise RuntimeError(
        "Dataset validation did not pass"
    )

print("\n================================")
print("CELL 7: PASS")
print("DATASET READY: TRUE")
print("No dataset files downloaded or copied")
print("================================")


=== STAGE 6 DATASET AUDIT ===
Dataset: /content/datasets/CICIoMT24/WINDOWS_CPLUS
Running validation in isolated Python process...

=== VALIDATION RESULTS ===
Status: PASS_METADATA_SCHEMA
CSV files: 322
Features: 380
Configurations: 4
Counts: {'FW_256_FS_8': {'train': 1348317, 'val': 449343, 'test': 449251}, 'FW_256_FS_16': {'train': 674169, 'val': 224677, 'test': 224631}, 'FW_512_FS_8': {'train': 1347101, 'val': 448735, 'test': 448563}, 'FW_512_FS_16': {'train': 673559, 'val': 224373, 'test': 224287}}
Label origin: SCENARIO_PROXY
Test feature rows read: 0
DATASET_READY: True

CELL 7: PASS
DATASET READY: TRUE
No dataset files downloaded or copied


In [24]:
# CELL 8 — Select exactly one experiment/config; default is DRY RUN
EXPERIMENT = "mi_rfe_rf"  #@param ["mi_rfe_rf", "baseline_svm", "mi_rfe_svm"]
FW = "256"  #@param [256, 512]
FS = 8  #@param [8, 16]
RUN_NAME = "mi_rfe_rf_FW256_FS8_run02"  #@param {type:"string"}
MODE = "full_train"  #@param ["dry_run", "full_train"]
CONFIRMATION = "RUN mi_rfe_rf FW_256_FS_8 mi_rfe_rf_FW256_FS8_run02"  #@param {type:"string"}
RUN_TECHNICAL_SVM_PROBE = False  #@param {type:"boolean"}
from colab.experiment_launcher import RunSpec
from colab.drive_manager import DRIVE_ROOT
SPEC = RunSpec(EXPERIMENT,f"FW_{FW}_FS_{FS}",RUN_NAME)
SPEC.validate()
LOCAL_RUNS = Path('/content/stage6_runs')
print({"experiment":SPEC.experiment,"config":SPEC.config,"run":SPEC.run_name,
       "mode":MODE,"local_output":str(LOCAL_RUNS),"drive_output":str(DRIVE_ROOT)})
print("Full train requires CELL 10 and the exact confirmation phrase.")

{'experiment': 'mi_rfe_rf', 'config': 'FW_256_FS_8', 'run': 'mi_rfe_rf_FW256_FS8_run02', 'mode': 'full_train', 'local_output': '/content/stage6_runs', 'drive_output': '/content/drive/MyDrive/COMPACT-MI-RFE'}
Full train requires CELL 10 and the exact confirmation phrase.


In [25]:

# CELL 9 — SAFE PREFLIGHT IN FRESH PYTHON PROCESS
# Avoid NumPy import errors in the Colab kernel
# No dataset copy, no download, no model.fit

from pathlib import Path
import subprocess
import sys
import json

print("=== STAGE 6 PREFLIGHT ===")

# 1. Verify prerequisites
if not DATASET_READY:
    raise RuntimeError(
        "Dataset validation has not passed."
    )

if not UNIT_TEST_EVIDENCE.get("passed", False):
    raise RuntimeError(
        "Unit tests have not passed."
    )

REPO = Path(REPO)
WINDOW_ROOT = Path(WINDOW_ROOT)
LOCAL_RUNS = Path(LOCAL_RUNS)
DRIVE_ROOT = Path(DRIVE_ROOT)

if not WINDOW_ROOT.is_dir():
    raise FileNotFoundError(WINDOW_ROOT)

# 2. Prepare experiment configuration
PROBE_PATH = LOCAL_RUNS / (
    SPEC.run_name + "_svm_technical_probe.json"
)

payload = {
    "repo": str(REPO),
    "windows": str(WINDOW_ROOT),
    "local_runs": str(LOCAL_RUNS),
    "drive_root": str(DRIVE_ROOT),
    "spec": {
        "experiment": SPEC.experiment,
        "config": SPEC.config,
        "run_name": SPEC.run_name,
    },
    "unit_test_evidence": UNIT_TEST_EVIDENCE,
    "svm_probe_path": (
        str(PROBE_PATH)
        if PROBE_PATH.exists()
        else None
    ),
}

# 3. Execute original preflight in fresh process
worker = r"""
import sys
import json
from pathlib import Path

data = json.load(sys.stdin)

repo = Path(data["repo"])
sys.path.insert(0, str(repo))

from colab.experiment_launcher import (
    RunSpec,
    preflight,
)

spec = RunSpec(**data["spec"])

probe = (
    Path(data["svm_probe_path"])
    if data["svm_probe_path"]
    else None
)

result = preflight(
    repo,
    Path(data["windows"]),
    Path(data["local_runs"]),
    Path(data["drive_root"]),
    spec,
    unit_test_evidence=data["unit_test_evidence"],
    svm_probe_path=probe,
)

print(
    "__PREFLIGHT_JSON__"
    + json.dumps(result, default=str)
)
"""

print("Python:", sys.version.split()[0])
print("Dataset:", WINDOW_ROOT)
print("Experiment:", SPEC.experiment)
print("Config:", SPEC.config)
print("Run:", SPEC.run_name)
print("\nRunning isolated preflight...")

process = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        worker,
    ],
    input=json.dumps(payload),
    cwd=str(REPO),
    capture_output=True,
    text=True,
    check=False,
)

# 4. Check worker execution
if process.returncode != 0:
    print(process.stdout[-4000:])
    print(process.stderr[-8000:])
    raise RuntimeError(
        "Preflight worker failed. "
        "No training was started."
    )

# 5. Recover preflight result
marker = "__PREFLIGHT_JSON__"

result_line = next(
    (
        line[len(marker):]
        for line in process.stdout.splitlines()
        if line.startswith(marker)
    ),
    None,
)

if result_line is None:
    raise RuntimeError(
        "Preflight worker did not return JSON."
    )

PREFLIGHT = json.loads(result_line)

# 6. Display results
print("\n=== PREFLIGHT RESULTS ===")

print("PREFLIGHT:", PREFLIGHT["status"])

print("\nCHECKS:")
for name, passed in PREFLIGHT["checks"].items():
    print(
        f"  {name}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nRAM GUARD:")
print(PREFLIGHT.get("ram_guard"))

print("\nLOCAL DISK REQUIRED BYTES:")
print(PREFLIGHT.get("local_disk_required_bytes"))

print("\nSAFETY:")
print(
    "Test feature rows read:",
    PREFLIGHT.get("test_feature_rows_read")
)
print(
    "model.fit calls:",
    PREFLIGHT.get("model_fit_calls")
)

# 7. Enforce preflight safety
if PREFLIGHT["status"] == "PASS":
    print("\nPREFLIGHT PASS")
else:
    failed = [
        key
        for key, value in PREFLIGHT["checks"].items()
        if not value
    ]
    print("\nPREFLIGHT FAIL")
    print("Failed checks:", failed)
    print("Full training is BLOCKED.")

print("\nCELL 9 COMPLETED")


=== STAGE 6 PREFLIGHT ===
Python: 3.13.15
Dataset: /content/datasets/CICIoMT24/WINDOWS_CPLUS
Experiment: mi_rfe_rf
Config: FW_256_FS_8
Run: mi_rfe_rf_FW256_FS8_run02

Running isolated preflight...

=== PREFLIGHT RESULTS ===
PREFLIGHT: PASS

CHECKS:
  repository_files: PASS
  python_supported: PASS
  selected_scientific_profile: PASS
  unit_tests_current_code: PASS
  approved_original_scripts: PASS
  dataset_manifest_schema: PASS
  no_existing_local_run: PASS
  no_existing_drive_run: PASS
  drive_mounted: PASS
  prior_fs16_not_rerun: PASS
  ram_admission: PASS
  local_disk_admission: PASS

RAM GUARD:
{'policy': 'stage54_empirical_scaled_FS16_peak_plus_max_2GiB_or_20pct', 'required_available_bytes': 25792855276, 'available_bytes': 52465545216, 'guard_is_not_proven_peak_bound': True}

LOCAL DISK REQUIRED BYTES:
8589934592

SAFETY:
Test feature rows read: 0
model.fit calls: 0

PREFLIGHT PASS

CELL 9 COMPLETED


In [ ]:

# CELL 10 — ISOLATED FULL TRAINING
# Fix NumPy kernel import error
# Keep all safety checks, checkpoints and Drive outputs
# No dataset download or copy

from pathlib import Path
import subprocess
import tempfile
import json
import sys

RUN_RESULT = None

REPO = Path(REPO).resolve()
WINDOW_ROOT = Path(WINDOW_ROOT).resolve()
LOCAL_RUNS = Path(LOCAL_RUNS)
DRIVE_ROOT = Path(DRIVE_ROOT)

PROBE_PATH = LOCAL_RUNS / (
    SPEC.run_name + "_svm_technical_probe.json"
)

print("=== STAGE 6 — OFFICIAL RUN ===")
print("Python:", sys.version.split()[0])
print("Experiment:", SPEC.experiment)
print("Config:", SPEC.config)
print("Run:", SPEC.run_name)
print("Mode:", MODE)
print("Dataset:", WINDOW_ROOT)

# 1. Safety checks
if not DATASET_READY:
    raise RuntimeError("Dataset validation has not passed")

if not UNIT_TEST_EVIDENCE.get("passed"):
    raise RuntimeError("Unit tests have not passed")

if not WINDOW_ROOT.is_dir():
    raise FileNotFoundError(WINDOW_ROOT)

SPEC.validate()

if MODE not in ("dry_run", "full_train"):
    raise ValueError("Unsupported training mode")

if RUN_TECHNICAL_SVM_PROBE and SPEC.experiment == "mi_rfe_rf":
    raise ValueError("SVM probe cannot be used for RF")

if MODE == "full_train":
    if PREFLIGHT["status"] != "PASS":
        raise RuntimeError("Preflight FAIL: training blocked")

    required = (
        f"RUN {SPEC.experiment} "
        f"{SPEC.config} {SPEC.run_name}"
    )

    if CONFIRMATION != required:
        raise PermissionError(
            f"Incorrect confirmation. Expected: {required}"
        )

    # Never overwrite earlier runs or checkpoints
    occupied = [
        LOCAL_RUNS / SPEC.run_name,
        LOCAL_RUNS / (SPEC.run_name + ".incomplete"),
        DRIVE_ROOT / "experiment_results" / SPEC.run_name,
        DRIVE_ROOT / "experiment_results" /
            (SPEC.run_name + ".incomplete"),
        DRIVE_ROOT / "checkpoints" /
            (SPEC.run_name + ".incomplete"),
        DRIVE_ROOT / "environment" /
            (SPEC.run_name + ".json"),
    ]

    existing = [str(p) for p in occupied if p.exists()]

    if existing:
        raise FileExistsError(
            "Run artifacts already exist; inspect them "
            "before choosing a new run name:\n"
            + "\n".join(existing)
        )

# 2. Pass configuration to the clean process
payload = {
    "repo": str(REPO),
    "windows": str(WINDOW_ROOT),
    "local_runs": str(LOCAL_RUNS),
    "drive_root": str(DRIVE_ROOT),
    "spec": {
        "experiment": SPEC.experiment,
        "config": SPEC.config,
        "run_name": SPEC.run_name,
    },
    "mode": MODE,
    "confirmation": CONFIRMATION,
    "unit_test_evidence": UNIT_TEST_EVIDENCE,
    "svm_probe": bool(RUN_TECHNICAL_SVM_PROBE),
    "probe_path": str(PROBE_PATH),
}

# 3. Fresh Python worker
worker = r"""
import json
import sys
from pathlib import Path

data = json.loads(
    Path(sys.argv[1]).read_text(encoding="utf-8")
)

repo = Path(data["repo"])
sys.path.insert(0, str(repo))

# Import scientific packages in fresh process
import numpy
import pandas
import sklearn
import scipy
import joblib

print("Scientific imports: PASS", flush=True)
print("NumPy:", numpy.__version__, flush=True)

from colab.experiment_launcher import (
    RunSpec,
    preflight,
    run_one,
    run_technical_svm_probe,
)

spec = RunSpec(**data["spec"])
spec.validate()

windows = Path(data["windows"])
local_runs = Path(data["local_runs"])
drive_root = Path(data["drive_root"])
probe_path = Path(data["probe_path"])

if data["svm_probe"]:
    local_runs.mkdir(parents=True, exist_ok=True)

    probe_result = run_technical_svm_probe(
        repo, windows, spec, probe_path
    )

    print(
        "__PROBE_JSON__" +
        json.dumps(probe_result, default=str),
        flush=True
    )

    audit = preflight(
        repo, windows, local_runs, drive_root, spec,
        unit_test_evidence=data["unit_test_evidence"],
        svm_probe_path=probe_path
    )

    print(
        "Updated preflight:",
        audit["status"],
        audit["checks"],
        flush=True
    )

    if data["mode"] == "full_train":
        raise RuntimeError(
            "SVM probe finished. Turn off probe, "
            "rerun Cell 9, then start full training."
        )

elif data["mode"] == "full_train":
    print("\nStarting official training...", flush=True)

    result = run_one(
        repo,
        windows,
        local_runs,
        drive_root,
        spec,
        unit_test_evidence=data["unit_test_evidence"],
        confirmation=data["confirmation"],
        svm_probe_path=(
            probe_path if probe_path.exists() else None
        ),
    )

    print(
        "__RUN_RESULT_JSON__" +
        json.dumps(result, default=str),
        flush=True
    )

else:
    print("Dry run only. No model.fit called.", flush=True)
"""

# 4. Launch worker and stream logs in real time
if MODE == "dry_run" and not RUN_TECHNICAL_SVM_PROBE:
    print("\nDRY RUN ONLY — No training started.")

else:
    print("\nLaunching isolated Python process...")
    print("Do not interrupt during full training.\n")

    with tempfile.TemporaryDirectory(
        prefix="stage6_launch_"
    ) as temp_dir:

        config_file = Path(temp_dir) / "launch.json"

        config_file.write_text(
            json.dumps(payload),
            encoding="utf-8"
        )

        process = subprocess.Popen(
            [
                sys.executable,
                "-u",
                "-c",
                worker,
                str(config_file),
            ],
            cwd=str(REPO),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            errors="replace",
        )

        result_json = None
        probe_json = None

        for line in process.stdout:
            clean = line.strip()

            if clean.startswith("__RUN_RESULT_JSON__"):
                result_json = clean.split(
                    "__RUN_RESULT_JSON__", 1
                )[1]
            elif clean.startswith("__PROBE_JSON__"):
                probe_json = clean.split(
                    "__PROBE_JSON__", 1
                )[1]
            else:
                print(line, end="", flush=True)

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Training worker failed (exit {return_code}). "
            "Inspect local and Drive checkpoints before "
            "retrying. No automatic restart is performed."
        )

    if result_json is not None:
        RUN_RESULT = json.loads(result_json)

        print("\n=== OFFICIAL RUN RESULT ===")
        print(json.dumps(
            RUN_RESULT,
            indent=2,
            ensure_ascii=False
        ))

        print("\nCELL 10: TRAINING COMPLETED")

    elif probe_json is not None:
        print("\nSVM TECHNICAL PROBE RESULT:")
        print(json.loads(probe_json))

    elif MODE == "full_train":
        raise RuntimeError(
            "Worker exited without an official result."
        )


=== STAGE 6 — OFFICIAL RUN ===
Python: 3.13.15
Experiment: mi_rfe_rf
Config: FW_256_FS_8
Run: mi_rfe_rf_FW256_FS8_run02
Mode: full_train
Dataset: /content/datasets/CICIoMT24/WINDOWS_CPLUS

Launching isolated Python process...
Do not interrupt during full training.

Scientific imports: PASS
NumPy: 2.2.6

Starting official training...
[0.0s] run_started stage=run_started rss=0.180GiB available=48.890GiB {"config": "FW_256_FS_8", "experiment": "mi_rfe_rf", "resume_supported": false}
[0.0s] split_load_started stage=split_load_started rss=0.180GiB available=48.890GiB {"split": "train", "expected_rows": 1348317, "files": 45}
[0.1s] source_file_loaded stage=source_file_loaded rss=0.185GiB available=48.888GiB {"split": "train", "source_id": 6, "source_rows": 82, "loaded_rows": 82, "expected_rows": 1348317}
[0.1s] source_file_loaded stage=source_file_loaded rss=0.188GiB available=48.885GiB {"split": "train", "source_id": 11, "source_rows": 44, "loaded_rows": 126, "expected_rows": 1348317}
[0.2s

In [ ]:
# CELL 11 — Verify publication; optionally promote an ALREADY COMPLETED checkpoint
import json
RECOVER_COMPLETED_CHECKPOINT = False  #@param {type:"boolean"}
if RECOVER_COMPLETED_CHECKPOINT:
    from colab.checkpoint_manager import publish_completed_checkpoint
    from colab.dataset_validator import EXPECTED
    recovered_path = publish_completed_checkpoint(DRIVE_ROOT,SPEC.run_name,EXPECTED[SPEC.config])
    print('Promoted existing completed checkpoint without fit or Test evaluation:',recovered_path)
if RUN_RESULT is None:
    print("No new official result to save; no Drive result folder created by this cell.")
else:
    result_dir = Path(RUN_RESULT['drive_result'])
    manifest = json.loads((result_dir/'artifact_manifest.json').read_text())
    for name, expected in manifest['files'].items():
        path = result_dir/name
        if path.stat().st_size != expected['bytes'] or sha256(path) != expected['sha256']:
            raise IOError(f"Drive result artifact failed verification: {name}")
    print("Drive artifacts verified:",result_dir)
    print("Environment:",DRIVE_ROOT/'environment'/(SPEC.run_name+'.json'))
    print("Checkpoints:",DRIVE_ROOT/'checkpoints'/(SPEC.run_name+'.incomplete'))
    print("No Test prediction/evaluation was repeated by this cell.")

In [ ]:
# CELL 12 — Read result.json only; never load Test CSV/model for summary
from colab.results_summary import completed_results, previous_artifacts_status
rows = completed_results(DRIVE_ROOT)
print("Previous FS16 artifacts:",previous_artifacts_status(DRIVE_ROOT))
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for row in rows: print(row)
print("SCENARIO_PROXY; Benign is rare; capture/session independence remains unproven.")